# 01 Data Understanding

Ziel dieses Notebooks ist es, die verwendeten CICIDS2017-Freitag-Dateien zu laden, ihre Struktur zu prüfen und die enthaltenen Labels sowie mögliche Datenprobleme zu analysieren.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw")

files = {
    "friday_morning": RAW_DATA_PATH / "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "friday_portscan": RAW_DATA_PATH / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "friday_ddos": RAW_DATA_PATH / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
}

dfs = {}

for name, path in files.items():
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    dfs[name] = df
    print(f"{name}: {df.shape}")

friday_morning: (191033, 85)
friday_portscan: (286467, 85)
friday_ddos: (225745, 85)


Die drei Freitag-Dateien wurden erfolgreich geladen. Die Spaltennamen wurden bereinigt, da CICIDS2017 teilweise Leerzeichen in den Spaltennamen enthält.

## Prüfung der Label-Verteilung

Im nächsten Schritt wird untersucht, welche Klassen in den drei Freitag-Dateien enthalten sind.

In [6]:
for name, df in dfs.items():
    print(f"\n{name}")
    print(df["Label"].value_counts())


friday_morning
Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

friday_portscan
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

friday_ddos
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


Die Label-Verteilung zeigt, welche Klassen in den einzelnen Dateien enthalten sind. Für das Projekt werden alle Labels außer BENIGN später als Angriff zusammengefasst.

In [7]:
for name, df in dfs.items():
    numeric_df = df.select_dtypes(include=[np.number])
    
    missing_values = df.isna().sum().sum()
    infinite_values = np.isinf(numeric_df).sum().sum()
    
    print(f"{name}")
    print(f"Fehlende Werte: {missing_values}")
    print(f"Infinity-Werte: {infinite_values}")
    print()

friday_morning
Fehlende Werte: 28
Infinity-Werte: 216

friday_portscan
Fehlende Werte: 15
Infinity-Werte: 727

friday_ddos
Fehlende Werte: 4
Infinity-Werte: 64



Die Prüfung der Datenqualität zeigt, dass alle drei Datensätze fehlende Werte und Infinity-Werte enthalten. Diese Werte müssen in der Datenaufbereitung behandelt werden, da sie beim späteren Training eines Machine-Learning-Modells zu Fehlern führen können.

## Identifikation betroffener Merkmale

Im nächsten Schritt wird untersucht, in welchen Spalten fehlende oder unendliche Werte auftreten. Dadurch kann entschieden werden, welche Merkmale in der Datenaufbereitung bereinigt oder entfernt werden müssen.

In [8]:
for name, df in dfs.items():
    print(f"\n{name}")

    # Fehlende Werte pro Spalte
    missing_per_column = df.isna().sum()
    missing_per_column = missing_per_column[missing_per_column > 0]

    print("Spalten mit fehlenden Werten:")
    print(missing_per_column)

    # Infinity-Werte pro numerischer Spalte
    numeric_df = df.select_dtypes(include=[np.number])
    inf_per_column = np.isinf(numeric_df).sum()
    inf_per_column = inf_per_column[inf_per_column > 0]

    print("\nSpalten mit Infinity-Werten:")
    print(inf_per_column)


friday_morning
Spalten mit fehlenden Werten:
Flow Bytes/s    28
dtype: int64

Spalten mit Infinity-Werten:
Flow Bytes/s       94
Flow Packets/s    122
dtype: int64

friday_portscan
Spalten mit fehlenden Werten:
Flow Bytes/s    15
dtype: int64

Spalten mit Infinity-Werten:
Flow Bytes/s      356
Flow Packets/s    371
dtype: int64

friday_ddos
Spalten mit fehlenden Werten:
Flow Bytes/s    4
dtype: int64

Spalten mit Infinity-Werten:
Flow Bytes/s      30
Flow Packets/s    34
dtype: int64


Die fehlenden und unendlichen Werte treten ausschließlich in den Merkmalen `Flow Bytes/s` und `Flow Packets/s` auf. Diese Merkmale basieren auf Verhältniswerten pro Sekunde und können bei sehr kurzen oder fehlerhaften Flows zu NaN- oder Infinity-Werten führen. In der Datenaufbereitung müssen diese Werte daher bereinigt werden.

## Zusammenführung der Freitag-Dateien

Im nächsten Schritt werden die drei Freitag-Dateien zu einem gemeinsamen Datensatz zusammengeführt. Dadurch entsteht die Grundlage für die spätere binäre Klassifikation von normalem Netzwerkverkehr und Angriffen.

In [9]:
df_all = pd.concat(dfs.values(), ignore_index=True)

print("Gesamtdatensatz:", df_all.shape)
print("\nLabel-Verteilung im Gesamtdatensatz:")
print(df_all["Label"].value_counts())

Gesamtdatensatz: (703245, 85)

Label-Verteilung im Gesamtdatensatz:
Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64


In [10]:
df_all["target"] = df_all["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

print("Binäre Zielvariable:")
print(df_all["target"].value_counts())

print("\n0 = BENIGN / normaler Traffic")
print("1 = Angriff")

Binäre Zielvariable:
target
0    414322
1    288923
Name: count, dtype: int64

0 = BENIGN / normaler Traffic
1 = Angriff


Die ursprünglichen Angriffstypen Bot, PortScan und DDoS wurden zu einer gemeinsamen Angriffsklasse zusammengefasst. Dadurch entsteht eine binäre Zielvariable für die spätere Klassifikation: BENIGN wird als normaler Netzwerkverkehr codiert, alle übrigen Labels als Angriff. Der resultierende Datensatz enthält 414.322 normale Netzwerk-Flows und 288.923 Angriffs-Flows.

## Prüfung der Spaltenstruktur

Im nächsten Schritt werden die vorhandenen Merkmale des kombinierten Datensatzes geprüft. Dabei wird insbesondere betrachtet, welche Spalten für die Modellierung relevant sind und welche Spalten potenziell entfernt werden sollten.

In [11]:
print("Anzahl Spalten:", len(df_all.columns))
print("\nSpalten:")
print(df_all.columns.tolist())

Anzahl Spalten: 86

Spalten:
['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Co

In [12]:
print("Datentypen:")
print(df_all.dtypes.value_counts())

print("\nNicht-numerische Spalten:")
print(df_all.select_dtypes(exclude=[np.number]).columns.tolist())

Datentypen:
int64      44
float64    37
str         5
Name: count, dtype: int64

Nicht-numerische Spalten:
['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Label']


In [15]:
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 703245 entries, 0 to 703244
Data columns (total 86 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Flow ID                      703245 non-null  str    
 1   Source IP                    703245 non-null  str    
 2   Source Port                  703245 non-null  int64  
 3   Destination IP               703245 non-null  str    
 4   Destination Port             703245 non-null  int64  
 5   Protocol                     703245 non-null  int64  
 6   Timestamp                    703245 non-null  str    
 7   Flow Duration                703245 non-null  int64  
 8   Total Fwd Packets            703245 non-null  int64  
 9   Total Backward Packets       703245 non-null  int64  
 10  Total Length of Fwd Packets  703245 non-null  int64  
 11  Total Length of Bwd Packets  703245 non-null  float64
 12  Fwd Packet Length Max        703245 non-null  int64  
 13  Fwd Packet

## Zusammenfassung Data Understanding

Im Rahmen des Data Understanding wurden die drei Freitag-Dateien des CICIDS2017-Datensatzes untersucht. Die Dateien enthalten insgesamt 703.245 Netzwerk-Flows mit 85 ursprünglichen Merkmalen. Enthalten sind normaler Netzwerkverkehr (BENIGN) sowie die Angriffstypen Bot, PortScan und DDoS.

Die Spaltenstruktur ist in allen drei Dateien identisch, sodass eine zeilenweise Zusammenführung möglich ist. Zusätzlich wurde festgestellt, dass fehlende Werte und Infinity-Werte ausschließlich in den Merkmalen `Flow Bytes/s` und `Flow Packets/s` auftreten. Für die weitere Modellierung wird eine binäre Zielvariable verwendet: BENIGN = 0 und alle Angriffstypen = 1.